In [1]:
!pip -q install torch transformers spacy nltk scikit-learn tqdm pandas numpy
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 25.2 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import os

# If you uploaded the whole repo to Colab, this usually works as-is.
# Otherwise update PROJECT_ROOT to wherever your project lives.
CANDIDATES = [
    Path('/content/pos-tagging-social-media'),
    Path('/content/drive/MyDrive/pos-tagging-social-media'),
    Path.cwd(),
]

PROJECT_ROOT = None
for candidate in CANDIDATES:
    if (candidate / 'data' / 'raw').exists() or (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
# Ensure MODEL_DIR always points to Google Drive from the start
# This overrides the previous logic which put it in /content
MODEL_DIR = Path("/content/drive/MyDrive/pos_models/distilbert_pos")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR exists =', RAW_DIR.exists())
print('MODEL_DIR =', MODEL_DIR)

PROJECT_ROOT = /content
RAW_DIR exists = False
MODEL_DIR = /content/drive/MyDrive/pos_models/distilbert_pos


In [4]:
# Essential configuration and device setup
import json
import logging
import random
import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import spacy
import torch
from nltk.tokenize import TweetTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('colab-pos')

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

@dataclass
class Config:
    model_name: str = 'distilbert-base-uncased'
    max_length: int = 64
    batch_size: int = 16
    epochs: int = 2
    learning_rate: float = 3e-5
    early_stopping_patience: int = 2
    max_train_samples: Optional[int] = None
    max_val_samples: Optional[int] = None

CFG = Config()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)
print(CFG)

DEVICE = cuda
Config(model_name='distilbert-base-uncased', max_length=64, batch_size=16, epochs=2, learning_rate=3e-05, early_stopping_patience=2, max_train_samples=None, max_val_samples=None)


In [5]:
# Utility functions for text cleaning and weak labeling (used by model.predict internally)
import emoji  # 🔥 add this

tweet_tokenizer = TweetTokenizer(reduce_len=True, preserve_case=False)
nlp = spacy.load('en_core_web_sm')

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
WS_RE = re.compile(r'\s+')


def clean_text(text: str) -> str:
    text = URL_RE.sub('', text)
    text = MENTION_RE.sub('', text)
    text = text.lower().strip()
    text = WS_RE.sub(' ', text)
    return text


# 🔥 UPDATED FUNCTION (IMPORTANT CHANGE HERE)
def weak_label_text(text: str) -> Tuple[List[str], List[str]]:
    doc = nlp(text)
    tokens = []
    labels = []

    for token in doc:
        tokens.append(token.text)

        # ✅ FIX: detect emoji → SYM
        if token.text in emoji.EMOJI_DATA:
            labels.append("SYM")
        else:
            labels.append(token.pos_ if token.pos_ else "X")

    return tokens, labels


def load_lines(path: Path, limit: Optional[int] = None) -> List[str]:
    lines = [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    return lines[:limit] if limit else lines


def build_dataset(raw_texts: List[str]) -> List[Dict[str, List[str]]]:
    dataset = []
    for text in tqdm(raw_texts, desc='Building weak labels'):
        cleaned = clean_text(text)
        if not cleaned:
            continue
        tokens, pos_tags = weak_label_text(cleaned)
        if not tokens or not pos_tags:
            continue
        dataset.append({'text': ' '.join(tokens), 'tokens': tokens, 'pos_tags': pos_tags})
    return dataset


# ✅ FIXED PATHS (important)
from pathlib import Path

train_path = Path('/content/train_text.txt')
val_path = Path('/content/val_text.txt')


if not train_path.exists() or not val_path.exists():
    raise FileNotFoundError(
        f'Missing dataset files. Expected: {train_path} and {val_path}. '
        'Upload the repo or mount Drive before running training.'
    )


train_texts = load_lines(train_path, CFG.max_train_samples)
val_texts = load_lines(val_path, CFG.max_val_samples)


if not train_texts or not val_texts:
    raise ValueError('Training/validation files are empty after loading.')


print(f'Loaded {len(train_texts)} training texts and {len(val_texts)} validation texts from raw files.')


train_data = build_dataset(train_texts)
val_data = build_dataset(val_texts)


all_labels = sorted({label for sample in train_data + val_data for label in sample['pos_tags']})
label_to_id = {label: idx for idx, label in enumerate(all_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}


print('train samples =', len(train_data))
print('val samples =', len(val_data))
print('labels =', label_to_id)

Loaded 45615 training texts and 2000 validation texts from raw files.


Building weak labels:   0%|          | 0/45615 [00:00<?, ?it/s]

Building weak labels:   0%|          | 0/2000 [00:00<?, ?it/s]

train samples = 45615
val samples = 2000
labels = {'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4, 'DET': 5, 'INTJ': 6, 'NOUN': 7, 'NUM': 8, 'PART': 9, 'PRON': 10, 'PROPN': 11, 'PUNCT': 12, 'SCONJ': 13, 'SYM': 14, 'VERB': 15, 'X': 16}


In [6]:
# Defines the OptimizedPOSTagger class and related functions (like tokenize_and_align_labels, predict)
import emoji

def is_emoji(s):
    return any(char in emoji.EMOJI_DATA for char in s)


def tokenize_and_align_labels(
    texts,
    labels,
    tokenizer,
    label_to_id,
    max_length=64,
):
    split_texts = [text.split() for text in texts]

    encodings = tokenizer(
        split_texts,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
    )

    aligned_labels = []

    for batch_index, word_labels in enumerate(labels):
        word_ids = encodings.word_ids(batch_index=batch_index)

        previous_word_idx = None
        label_ids = []

        for token_idx, word_idx in enumerate(word_ids):

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                if word_idx < len(word_labels):
                    label_ids.append(
                        label_to_id.get(word_labels[word_idx], label_to_id.get('X', 0))
                    )
                else:
                    label_ids.append(-100)

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    encodings.pop("offset_mapping", None)
    encodings["labels"] = aligned_labels

    return encodings


class OptimizedTokenDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            key: torch.tensor(value[idx], dtype=torch.long)
            for key, value in self.encodings.items()
        }


class OptimizedPOSTagger:
    def __init__(self, label_to_id, id_to_label, config: Config):
        self.label_to_id = label_to_id
        self.id_to_label = id_to_label
        self.config = config

        self.tokenizer = AutoTokenizer.from_pretrained(
            config.model_name,
            use_fast=True
        )

        self.model = AutoModelForTokenClassification.from_pretrained(
            config.model_name,
            num_labels=len(label_to_id),
            label2id=label_to_id,
            id2label=id_to_label,
        )

        # Freeze embeddings
        for param in self.model.distilbert.embeddings.parameters():
            param.requires_grad = False

        # Freeze early layers
        for i, layer in enumerate(self.model.distilbert.transformer.layer):
            if i < 3:
                for param in layer.parameters():
                    param.requires_grad = False

        self.data_collator = DataCollatorForTokenClassification(
            self.tokenizer,
            padding=True
        )

        self.model.to(DEVICE)

    def create_dataset(self, texts: List[str], labels: List[List[str]]):
        encodings = tokenize_and_align_labels(
            texts,
            labels,
            self.tokenizer,
            self.label_to_id,
            self.config.max_length
        )
        return OptimizedTokenDataset(encodings)

    def create_dataloader(self, dataset, shuffle=False):
        return DataLoader(
            dataset,
            batch_size=self.config.batch_size,
            shuffle=shuffle,
            num_workers=0,   # 🔥 FIXED (avoid colab multiprocessing bug)
            pin_memory=False,
            collate_fn=self.data_collator,
        )

    # 🔥 FINAL FIXED PREDICT FUNCTION
    def predict(self, text: str):
        self.model.eval()

        words = clean_text(text).split()

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.config.max_length,
            return_tensors='pt',
        )

        encoding = {k: v.to(DEVICE) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = self.model(**encoding)

        preds = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()

        word_ids = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.config.max_length
        ).word_ids()

        seen = set()
        tagged = []

        for token_idx, word_idx in enumerate(word_ids):
            if word_idx is None or word_idx in seen or word_idx >= len(words):
                continue

            seen.add(word_idx)

            word = words[word_idx]
            label = self.id_to_label[preds[token_idx]]

            # 🔥 EMOJI OVERRIDE FIX
            if is_emoji(word):
                label = "SYM"

            tagged.append((word, label))

        return tagged
def compute_metrics(eval_predictions, eval_labels):
    accuracy = accuracy_score(eval_labels, eval_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        eval_labels,
        eval_predictions,
        average='weighted',
        zero_division=0,
    )
    return {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
    }


def train_model(model_wrapper, train_loader, val_loader, config: Config):
    optimizer = AdamW(model_wrapper.model.parameters(), lr=config.learning_rate)
    best_f1 = float('-inf')
    patience = 0
    history = {'train_loss': [], 'eval_loss': [], 'eval_f1': [], 'eval_accuracy': []}

    for epoch in range(config.epochs):
        model_wrapper.model.train()
        total_loss = 0.0
        train_bar = tqdm(train_loader, desc=f'Train epoch {epoch + 1}/{config.epochs}')

        for batch in train_bar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model_wrapper.model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            train_bar.set_postfix(loss=f'{loss.item():.4f}')

        avg_train_loss = total_loss / max(len(train_loader), 1)
        history['train_loss'].append(avg_train_loss)

        model_wrapper.model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc='Validate'):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                outputs = model_wrapper.model(**batch)
                val_loss += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=-1)
                labels = batch['labels']
                mask = labels != -100
                all_preds.extend(preds[mask].cpu().tolist())
                all_labels.extend(labels[mask].cpu().tolist())

        metrics = compute_metrics(all_preds, all_labels)
        avg_val_loss = val_loss / max(len(val_loader), 1)
        history['eval_loss'].append(avg_val_loss)
        history['eval_f1'].append(metrics['f1'])
        history['eval_accuracy'].append(metrics['accuracy'])

        print({
            'epoch': epoch + 1,
            'train_loss': round(avg_train_loss, 4),
            'val_loss': round(avg_val_loss, 4),
            'accuracy': round(metrics['accuracy'], 4),
            'f1': round(metrics['f1'], 4),
        })

        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            patience = 0
            model_wrapper.model.save_pretrained(MODEL_DIR)
            model_wrapper.tokenizer.save_pretrained(MODEL_DIR)
            with open(MODEL_DIR / 'colab_config.json', 'w', encoding='utf-8') as f:
                json.dump({
                    'label_to_id': label_to_id,
                    'id_to_label': {str(k): v for k, v in id_to_label.items()},
                    'max_length': config.max_length,
                    'model_name': config.model_name,
                }, f, indent=2)
        else:
            patience += 1
            if patience >= config.early_stopping_patience:
                print('Early stopping triggered.')
                break

    return history

In [7]:
model_wrapper = OptimizedPOSTagger(label_to_id, id_to_label, CFG)

train_dataset = model_wrapper.create_dataset(
    [sample['text'] for sample in train_data],
    [sample['pos_tags'] for sample in train_data],
)
val_dataset = model_wrapper.create_dataset(
    [sample['text'] for sample in val_data],
    [sample['pos_tags'] for sample in val_data],
)

train_loader = model_wrapper.create_dataloader(train_dataset, shuffle=True)
val_loader = model_wrapper.create_dataloader(val_dataset, shuffle=False)

history = train_model(model_wrapper, train_loader, val_loader, CFG)
history

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Train epoch 2/2:   0%|          | 0/2851 [00:00<?, ?it/s]

Validate:   0%|          | 0/125 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.215, 'val_loss': 0.2042, 'accuracy': 0.9321, 'f1': 0.9317}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_loss': [0.3225325379521961, 0.2149927448213456],
 'eval_loss': [0.22176349657773972, 0.20415660619735718],
 'eval_f1': [0.926704498153899, 0.9317226740460588],
 'eval_accuracy': [0.9271052170868347, 0.9321165966386554]}

In [8]:
from pathlib import Path
import json

MODEL_DIR = Path('/content/drive/MyDrive/pos_models/distilbert_pos')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_wrapper.model.save_pretrained(MODEL_DIR)
model_wrapper.tokenizer.save_pretrained(MODEL_DIR)

with open(MODEL_DIR / "labels.json", "w") as f:
    json.dump(label_to_id, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
import os
print(os.listdir('/content/drive/MyDrive/pos_models/distilbert_pos'))

['colab_config.json', 'model.safetensors', 'tokenizer.json', 'labels.json', 'config.json', 'tokenizer_config.json']


In [11]:
print(model_wrapper.id_to_label)


{0: 'ADJ', 1: 'ADP', 2: 'ADV', 3: 'AUX', 4: 'CCONJ', 5: 'DET', 6: 'INTJ', 7: 'NOUN', 8: 'NUM', 9: 'PART', 10: 'PRON', 11: 'PROPN', 12: 'PUNCT', 13: 'SCONJ', 14: 'SYM', 15: 'VERB', 16: 'X'}


In [12]:
print(model_wrapper.predict("💀 😂 🔥"))

[('💀', 'SYM'), ('😂', 'SYM'), ('🔥', 'SYM')]


In [13]:
print('Saved model files:')
for path in sorted(MODEL_DIR.glob('*')):
    print('-', path.name)

examples = [
   'bro this was insane fr',
'nah this ain t it 💀',
'yo that match was crazyyyy',
'idk what you are talking about tbh',
'this is kinda sus ngl',
]

for text in examples:
    print('\nTEXT:', text)
    print(model_wrapper.predict(text))

Saved model files:
- colab_config.json
- config.json
- labels.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json

TEXT: bro this was insane fr
[('bro', 'PROPN'), ('this', 'PRON'), ('was', 'AUX'), ('insane', 'ADJ'), ('fr', 'NOUN')]

TEXT: nah this ain t it 💀
[('nah', 'INTJ'), ('this', 'PRON'), ('ain', 'VERB'), ('t', 'NOUN'), ('it', 'PRON'), ('💀', 'SYM')]

TEXT: yo that match was crazyyyy
[('yo', 'PROPN'), ('that', 'DET'), ('match', 'NOUN'), ('was', 'AUX'), ('crazyyyy', 'NOUN')]

TEXT: idk what you are talking about tbh
[('idk', 'INTJ'), ('what', 'PRON'), ('you', 'PRON'), ('are', 'AUX'), ('talking', 'VERB'), ('about', 'ADP'), ('tbh', 'NOUN')]

TEXT: this is kinda sus ngl
[('this', 'PRON'), ('is', 'AUX'), ('kinda', 'ADV'), ('sus', 'NOUN'), ('ngl', 'PROPN')]


In [14]:
test_path = Path('/content/test_text.txt')

test_texts = load_lines(test_path)
test_data = build_dataset(test_texts)

test_dataset = model_wrapper.create_dataset(
    [sample['text'] for sample in test_data],
    [sample['pos_tags'] for sample in test_data],
)

test_loader = model_wrapper.create_dataloader(test_dataset, shuffle=False)

Building weak labels:   0%|          | 0/12284 [00:00<?, ?it/s]

In [15]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def evaluate_model(model_wrapper, dataloader):
    model_wrapper.model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            outputs = model_wrapper.model(**batch)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=-1)
            labels = batch["labels"]

            preds = preds.cpu().numpy()
            labels = labels.cpu().numpy()

            for pred_seq, label_seq in zip(preds, labels):
                for p, l in zip(pred_seq, label_seq):
                    if l != -100:   # ignore padding
                        all_preds.append(p)
                        all_labels.append(l)

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted'
    )

    return {
        "accuracy": round(accuracy, 4),
        "f1": round(f1, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4)
    }

In [16]:
test_metrics = evaluate_model(model_wrapper, test_loader)
print("TEST RESULTS:", test_metrics)

TEST RESULTS: {'accuracy': 0.9035, 'f1': 0.9029, 'precision': 0.9035, 'recall': 0.9035}


### Loading the Saved Model from Google Drive and Making Predictions

In [17]:
# Loads the saved model configuration, tokenizer, and weights from Google Drive
import json
from transformers import AutoTokenizer, AutoModelForTokenClassification
from pathlib import Path

# Define MODEL_DIR (needs to be consistent with where it was saved)
MODEL_DIR = Path("/content/drive/MyDrive/pos_models/distilbert_pos")

# Load the configuration used during training
with open(MODEL_DIR / 'colab_config.json', 'r', encoding='utf-8') as f:
    loaded_config = json.load(f)

# Recreate label_to_id and id_to_label from the loaded config
loaded_label_to_id = loaded_config['label_to_id']
loaded_id_to_label = {int(k): v for k, v in loaded_config['id_to_label'].items()}

# Create a new Config object with loaded parameters
loaded_CFG = Config(
    model_name=loaded_config['model_name'],
    max_length=loaded_config['max_length']
    # Other config parameters like batch_size, epochs etc. are not needed for inference here
)

# Re-initialize model_wrapper with the saved model for inference
# We pass the loaded label_to_id, id_to_label, and loaded_CFG
model_wrapper_loaded = OptimizedPOSTagger(loaded_label_to_id, loaded_id_to_label, loaded_CFG)

# Load the fine-tuned tokenizer and model from MODEL_DIR
model_wrapper_loaded.tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model_wrapper_loaded.model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

# Ensure the model is on the correct device
model_wrapper_loaded.model.to(DEVICE)

print("Model and tokenizer successfully reloaded for inference from Drive.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model and tokenizer successfully reloaded for inference from Drive.


In [18]:
# Demonstrates how to use the loaded model for prediction on a custom text
custom_text = "This is a custom sentence to test the model with emojis! 👍😊"

print(f"Custom Text: {custom_text}")
print(model_wrapper_loaded.predict(custom_text))

Custom Text: This is a custom sentence to test the model with emojis! 👍😊
[('this', 'PRON'), ('is', 'AUX'), ('a', 'DET'), ('custom', 'ADJ'), ('sentence', 'NOUN'), ('to', 'PART'), ('test', 'VERB'), ('the', 'DET'), ('model', 'NOUN'), ('with', 'ADP'), ('emojis!', 'NOUN'), ('👍😊', 'SYM')]


In [5]:
import shutil
from pathlib import Path
from google.colab import files

SOURCE_DIR = Path("/content/drive/MyDrive/pos_models/distilbert_pos")

# Zip file path (same style as yours)
zip_path = shutil.make_archive(
    base_name="/content/distilbert_pos",   # zip will be created in /content
    format="zip",
    root_dir=str(SOURCE_DIR),
)

print(f"Zipped to: {zip_path}")
print(f"Size: {Path(zip_path).stat().st_size / 1e6:.1f} MB")

# Download
files.download(zip_path)

print("Download started.")

Zipped to: /content/distilbert_pos.zip
Size: 0.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.


In [15]:
from google.colab import files

uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

print("Uploaded file:", zip_path)

Saving distilbert_pos.zip to distilbert_pos (2).zip
Uploaded file: distilbert_pos (2).zip


In [16]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_list = zip_ref.namelist()

print("Files inside ZIP:", file_list)

Files inside ZIP: ['colab_config.json', 'model.safetensors', 'tokenizer.json', 'labels.json', 'config.json', 'tokenizer_config.json']


In [17]:
import os

extract_path = "/content/model"

# create folder
os.makedirs(extract_path, exist_ok=True)

# extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully!")
print("Contents:", os.listdir(extract_path))

Unzipped successfully!
Contents: ['colab_config.json', 'model.safetensors', 'tokenizer.json', 'labels.json', 'config.json', 'tokenizer_config.json']


In [18]:
model_path = extract_path

# go inside if single folder exists
while len(os.listdir(model_path)) == 1:
    model_path = os.path.join(model_path, os.listdir(model_path)[0])

print("Final model path:", model_path)
print("Files inside:", os.listdir(model_path))

Final model path: /content/model
Files inside: ['colab_config.json', 'model.safetensors', 'tokenizer.json', 'labels.json', 'config.json', 'tokenizer_config.json']


In [19]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)

try:
    # Full model
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    print("✅ Loaded FULL model")

except:
    # Adapter model (LoRA)
    from peft import PeftModel

    base_model = AutoModelForTokenClassification.from_pretrained("distilbert-base-uncased")
    model = PeftModel.from_pretrained(base_model, model_path)

    print("✅ Loaded ADAPTER model")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

✅ Loaded FULL model


In [22]:
import torch

model.eval()
id2label = model.config.id2label

while True:
    text = input("\nEnter a sentence (or type 'exit' to quit): ")

    if text.lower() == "exit":
        print("Exiting... 👋")
        break

    # Tokenize with word alignment
    words = text.split()
    inputs = tokenizer(words, return_tensors="pt", is_split_into_words=True)

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    word_ids = inputs.word_ids()
    labels = predictions[0].tolist()

    print("\n🔍 POS Tags:")

    current_word = None
    current_labels = []

    for word_id, label_id in zip(word_ids, labels):
        if word_id is None:
            continue

        if word_id != current_word:
            if current_labels:
                final_label = id2label[max(set(current_labels), key=current_labels.count)]
                print(f"{words[current_word]:15} → {final_label}")
            current_word = word_id
            current_labels = [label_id]
        else:
            current_labels.append(label_id)

    # last word
    if current_labels:
        final_label = id2label[max(set(current_labels), key=current_labels.count)]
        print(f"{words[current_word]:15} → {final_label}")


Enter a sentence (or type 'exit' to quit): Hi Vamsi

🔍 POS Tags:
Hi              → INTJ
Vamsi           → NOUN

Enter a sentence (or type 'exit' to quit): Hi Prachi 

🔍 POS Tags:
Hi              → INTJ
Prachi          → PROPN

Enter a sentence (or type 'exit' to quit): Wassup

🔍 POS Tags:
Wassup          → NOUN

Enter a sentence (or type 'exit' to quit): Fuck

🔍 POS Tags:
Fuck            → NOUN

Enter a sentence (or type 'exit' to quit): Fuck you

🔍 POS Tags:
Fuck            → NOUN
you             → PRON

Enter a sentence (or type 'exit' to quit): I'm doing great

🔍 POS Tags:
I'm             → AUX
doing           → VERB
great           → ADJ

Enter a sentence (or type 'exit' to quit): I am doing good

🔍 POS Tags:
I               → PRON
am              → AUX
doing           → VERB
good            → ADJ

Enter a sentence (or type 'exit' to quit): See you later

🔍 POS Tags:
See             → VERB
you             → PRON
later           → ADV

Enter a sentence (or type 'exit' to quit): exi